# 04_Gold_Aggregations_Optimization
This notebook reads the curated Silver tables, builds business‑ready Gold tables, and applies Delta Lake performance optimizations (OPTIMIZE, Z‑ORDER, COALESCE).
All tables are stored in the `ecommerce_catalog.gold` schema.


In [0]:
# ============================================================
# GOLD LAYER
# ============================================================
# Source:
#   samplework.silver.orders
#
# Gold tables:
#   1. daily_sales
#   2. customer_sales
#   3. product_sales
#   4. top_customers
#
# Features:
#   - Business aggregations
#   - Window functions
#   - coalesce()
#   - OPTIMIZE
#   - ZORDER
#   - Unity Catalog
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# CONFIGURATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "samplework"

orders = spark.table(f"{CATALOG}.silver.orders")
customers = spark.table(f"{CATALOG}.silver.customers")
products = spark.table(f"{CATALOG}.silver.products")



In [0]:
display(spark.table("samplework.silver.customers"))

In [0]:
display(spark.table("samplework.silver.products"))

In [0]:
display(spark.table("samplework.silver.orders"))

```md
                    ┌─────────────────┐
                    │   dim_customer  │
                    │─────────────────│
                    │ customer_id     │
                    │ customer_name   │
                    │ email           │
                    │ city            │
                    │ age             │
                    └────────┬────────┘
                             │
                             │
┌─────────────────┐          ▼          ┌─────────────────┐
│  dim_product    │──────► fact_orders ◄──────dim_date   │
│─────────────────│          │           │────────────────│
│ product_id      │          │           │ date_key       │
│ product_name    │          │           │ date           │
│ category        │          │           │ year           │
│ price           │          │           │ month          │
└─────────────────┘          │           │ day            │
                             └───────────┴────────────────┘
```

In [0]:
dim_customer = (
    customers
    .select(
        "customer_id",
        F.col("name").alias("customer_name"),
        "email",
        "city",
        "age"
    )
    .dropDuplicates(["customer_id"])
)

display(dim_customer)

In [0]:
(
    dim_customer
    .coalesce(2)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.gold.dim_customer")
)


In [0]:
dim_product = (
    products
    .select(
        "product_id",
        "product_name",
        "category",
        "price",
        "stock"
    )
    .dropDuplicates(["product_id"])
)

display(dim_product)

In [0]:
(
    dim_product
    .coalesce(2)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.gold.dim_product")
)

In [0]:
dim_date = (
    orders
    .select("order_date")
    .filter(F.col("order_date").isNotNull())
    .distinct()
    .withColumn(
        "date_key",
        F.date_format("order_date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("order_date"))
    .withColumn("quarter", F.quarter("order_date"))
    .withColumn("month", F.month("order_date"))
    .withColumn("month_name", F.date_format("order_date", "MMMM"))
    .withColumn("day", F.dayofmonth("order_date"))
    .withColumn("day_name", F.date_format("order_date", "EEEE"))
)

display(dim_date)

In [0]:
(
    dim_date
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.gold.dim_date")
)

In [0]:
product_price = (
    products
    .select(
        "product_id",
        F.col("price").alias("unit_price")
    )
    .dropDuplicates(["product_id"])
)

In [0]:
fact_orders = (
    orders
    .join(
        product_price,
        on="product_id",
        how="left"
    )
    .withColumn(
        "date_key",
        F.date_format(
            "order_date",
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "total_amount",
        F.col("quantity") * F.col("unit_price")
    )
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "date_key",
        "order_date",
        "quantity",
        "unit_price",
        "total_amount",
        "status"
    )
)

display(fact_orders)

In [0]:
(
    fact_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.gold.fact_orders")
)

print("Gold Star Schema created successfully!")

1. Total Orders

In [0]:
%sql
SELECT
    COUNT(*) AS total_orders
FROM samplework.silver.orders;

Total Completed Orders

In [0]:
%sql
SELECT
    COUNT(*) AS completed_orders
FROM samplework.silver.orders
WHERE status = 'COMPLETED';

Total Revenue

In [0]:
%sql
SELECT
    SUM(o.quantity * p.price) AS total_revenue
FROM samplework.silver.orders o
JOIN samplework.silver.products p
    ON o.product_id = p.product_id
WHERE o.status = 'COMPLETED';

Delhi ke customers ne Electronics category mein kitna revenue generate kiya?

In [0]:
%sql
SELECT
    SUM(f.total_amount) AS revenue
FROM samplework.gold.fact_orders f
JOIN samplework.gold.dim_customer c
    ON f.customer_id = c.customer_id
JOIN samplework.gold.dim_product p
    ON f.product_id = p.product_id
WHERE c.city = 'Delhi'
  AND p.category = 'ELECTRONICS';